**复现入口**：新分析请优先使用仓库根目录命令行，参数与过程记录见 `config/default.yaml` 与 `results/GSE146912_run/provenance/`。

```bash
python scripts/run_pipeline.py --config config/default.yaml --stages main pec_subcluster
```



In [1]:
import os
import shutil
import scanpy as sc

# 项目根目录（请在迁移后的仓库根目录打开本 notebook）
PROJECT = os.path.abspath(".")

# 输出写在同盘 results 下；若需单独指定根路径可设置环境变量 SCRNA_OUTPUT_DIR
OUTPUT_ROOT = os.environ.get("SCRNA_OUTPUT_DIR", os.path.join(PROJECT, "results"))
OUT_DIR = os.path.join(OUTPUT_ROOT, "GSE146912_run")
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")
for d in (OUT_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

du = shutil.disk_usage(OUT_DIR)
print(f"输出目录: {OUT_DIR}\n该分区剩余: {du.free / (1024**3):.1f} GiB")

DATA_PATH = os.path.join(PROJECT, "results", "GSE146912_Merged_Raw.h5ad")

sc.settings.figdir = FIG_DIR
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)

print("数据路径:", DATA_PATH)
adata = sc.read_h5ad(DATA_PATH)
print(adata)

xmax = 0.0
step = 8000
for i in range(0, adata.n_obs, step):
    chunk = adata.X[i : i + step]
    if hasattr(chunk, "data") and chunk.data.size:
        xmax = max(xmax, float(chunk.data.max()))
print(f"X 非零元素全局最大值约 {xmax:.3f}")
if xmax < 100:
    print("视为已完成 per-cell 缩放/对数类变换，跳过 normalize_total 与 log1p。")
else:
    print("按原始计数流程：normalize_total + log1p")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

adata.raw = adata.copy()

输出目录: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run
该分区剩余: 699.3 GiB
数据路径: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_Merged_Raw.h5ad
AnnData object with n_obs × n_vars = 81648 × 52636
    obs: 'Sample', 'n_genes', 'batch'
X 非零元素全局最大值约 12.913
视为已完成 per-cell 缩放/对数类变换，跳过 normalize_total 与 log1p。


In [2]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    batch_key="batch",
    flavor="seurat",
    subset=True,
)
sc.pp.scale(adata, max_value=10, zero_center=False)
sc.tl.pca(adata, svd_solver="arpack")
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5, key_added="leiden", flavor="igraph", n_iterations=2)
print("聚类完成，类别数:", adata.obs["leiden"].nunique())

extracting highly variable genes
    finished (0:00:15)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)
... be careful when using `max_value` without `zero_center`.
computing PCA
    with n_comps=50
    finished (0:00:31)
computing neighbors
    using 'X_pca' with n_pcs = 40


/home/inspur/anaconda3/envs/scrna_py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:35)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:01:06)
running Leiden clustering
    finished: found 30 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:04)
聚类完成，类别数: 30


In [3]:
import matplotlib.pyplot as plt

sc.pl.umap(
    adata,
    color=["Sample", "leiden"],
    wspace=0.35,
    title=["Sample (batch)", "Leiden"],
    save="_merged_sample_leiden.pdf",
    show=False,
)
plt.show()

In [4]:
import mygene

marker_symbols = {
    "POD (足细胞)": ["Nphs1", "Nphs2", "Thsd7a", "Pdpn", "Wt1", "Mafb"],
    "PEC (壁层上皮)": ["Cldn1", "Dkk3", "Akap12", "Pax8", "Pax2", "Krt8", "Krt18"],
    "TEC (肾小管上皮)": ["Fxyd2", "Umod", "Aqp2", "Aqp3", "Slc12a1", "Slc12a3"],
    "END (内皮)": ["Flt1", "Tie1", "Pecam1", "Kdr"],
    "MES (系膜)": ["Gata3", "Pdgfrb", "Des", "Itga8"],
    "SMC (平滑肌)": ["Acta2", "Ren1", "Myh11", "Tagln"],
    "IMM (免疫)": ["Cd68", "Cd14", "Ptprc", "Cd3d"],
}

flat = sorted({g for genes in marker_symbols.values() for g in genes})
mg = mygene.MyGeneInfo()
res = mg.querymany(flat, scopes="symbol", fields="ensembl.gene", species="mouse", verbose=False)
sym_to_ens = {}
for r in res:
    q = r.get("query")
    eg = r.get("ensembl")
    if isinstance(eg, list) and eg:
        eg = eg[0]
    if isinstance(eg, dict):
        gid = eg.get("gene")
        if gid:
            sym_to_ens[q] = gid

marker_ens = {k: [sym_to_ens[s] for s in v if s in sym_to_ens] for k, v in marker_symbols.items()}
missing = set(flat) - set(sym_to_ens)
if missing:
    print("未映射到 Ensembl:", missing)

sc.pl.dotplot(
    adata,
    marker_ens,
    groupby="leiden",
    use_raw=True,
    standard_scale="var",
    dendrogram=True,
    save="_kidney_markers_leiden.pdf",
    show=False,
)
import matplotlib.pyplot as plt
plt.show()

    using 'X_pca' with n_pcs = 50
Storing dendrogram info using `.uns['dendrogram_leiden']`
categories: 0, 1, 2, etc.
var_group_labels: POD (足细胞), PEC (壁层上皮), TEC (肾小管上皮), etc.


/home/inspur/anaconda3/envs/scrna_py312/lib/python3.12/site-packages/scanpy/plotting/_utils.py:320: UserWarning: Glyph 36275 (\N{CJK UNIFIED IDEOGRAPH-8DB3}) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=dpi, bbox_inches="tight")
/home/inspur/anaconda3/envs/scrna_py312/lib/python3.12/site-packages/scanpy/plotting/_utils.py:320: UserWarning: Glyph 32454 (\N{CJK UNIFIED IDEOGRAPH-7EC6}) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=dpi, bbox_inches="tight")
/home/inspur/anaconda3/envs/scrna_py312/lib/python3.12/site-packages/scanpy/plotting/_utils.py:320: UserWarning: Glyph 32990 (\N{CJK UNIFIED IDEOGRAPH-80DE}) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=dpi, bbox_inches="tight")
/home/inspur/anaconda3/envs/scrna_py312/lib/python3.12/site-packages/scanpy/plotting/_utils.py:320: UserWarning: Glyph 22721 (\N{CJK UNIFIED IDEOGRAPH-58C1}) missing from font(s) DejaVu Sans.
  plt.savefig(filename, dpi=dpi, bbox_inches="tight")
/home/inspur/ana

## 2. 主要细胞类型注释（POD / PEC / TEC / END / MES / SMC / IMM）

基于文献标志基因模块打分（`score_genes`，`use_raw=True` 使用全转录组），按最高分类型赋 `cell_type_major`。随后**仅保留 PEC**，在**全基因** `raw` 空间内重新选高变基因、降维与 Leiden 聚类，做亚群差异基因与 **GO（Enrichr，小鼠）** 富集，并结合足细胞/肾小管模块分数讨论谱系倾向。

**说明**：若仅运行本节，请先执行上方数据加载与 `adata.raw` 构建（Cell 0–1），保证 `adata.raw` 含全部基因。

In [5]:
import mygene
import pandas as pd
import numpy as np

marker_symbols_major = {
    "POD": ["Nphs1", "Nphs2", "Thsd7a", "Pdpn", "Wt1", "Mafb"],
    "PEC": ["Cldn1", "Dkk3", "Akap12", "Pax8", "Pax2", "Krt8", "Krt18"],
    "TEC": ["Fxyd2", "Umod", "Aqp2", "Aqp3", "Slc12a1", "Slc12a3"],
    "END": ["Flt1", "Tie1", "Pecam1", "Kdr"],
    "MES": ["Gata3", "Pdgfrb", "Des", "Itga8"],
    "SMC": ["Acta2", "Ren1", "Myh11", "Tagln"],
    "IMM": ["Cd68", "Cd14", "Ptprc", "Cd3d"],
}

flat_m = sorted({g for genes in marker_symbols_major.values() for g in genes})
mg_ann = mygene.MyGeneInfo()
res_m = mg_ann.querymany(
    flat_m, scopes="symbol", fields="ensembl.gene", species="mouse", verbose=False
)
sym_to_ens_m = {}
for r in res_m:
    q = r.get("query")
    eg = r.get("ensembl")
    if isinstance(eg, list) and eg:
        eg = eg[0]
    if isinstance(eg, dict):
        gid = eg.get("gene")
        if gid:
            sym_to_ens_m[q] = gid

missing_m = set(flat_m) - set(sym_to_ens_m)
if missing_m:
    print("未映射到 Ensembl（主要类型）:", missing_m)

for ctype, genes in marker_symbols_major.items():
    ens_ids = [sym_to_ens_m[g] for g in genes if g in sym_to_ens_m]
    ens_ids = [e for e in ens_ids if e in adata.raw.var_names]
    if len(ens_ids) < 2:
        print(f"警告: {ctype} 可用基因过少 ({len(ens_ids)})，跳过打分")
        continue
    sc.tl.score_genes(adata, gene_list=ens_ids, score_name=f"score_{ctype}", use_raw=True)

score_cols = [f"score_{k}" for k in marker_symbols_major if f"score_{k}" in adata.obs]
score_mat = adata.obs[score_cols].copy()
adata.obs["cell_type_major"] = score_mat.idxmax(axis=1).str.replace("score_", "")
adata.obs["cell_type_max_score"] = score_mat.max(axis=1)
adata.obs["cell_type_second_score"] = score_mat.apply(
    lambda r: float(np.sort(r.values)[-2]), axis=1
)
adata.obs["cell_type_score_margin"] = (
    adata.obs["cell_type_max_score"] - adata.obs["cell_type_second_score"]
)
#  margin 过小可标为低置信度（可选）
adata.obs["cell_type_major_filter"] = adata.obs["cell_type_major"]
adata.obs.loc[adata.obs["cell_type_score_margin"] < 0.03, "cell_type_major_filter"] = "Low_confidence"

print(adata.obs["cell_type_major"].value_counts())
print("\n与 Leiden 交叉表（前 15 行）:")
print(
    pd.crosstab(adata.obs["leiden"], adata.obs["cell_type_major"])
    .head(15)
    .to_string()
)

computing score 'score_POD'
    finished: added
    'score_POD', score of gene set (adata.obs).
    99 total control genes are used. (0:00:08)
computing score 'score_PEC'
    finished: added
    'score_PEC', score of gene set (adata.obs).
    200 total control genes are used. (0:00:07)
computing score 'score_TEC'
    finished: added
    'score_TEC', score of gene set (adata.obs).
    250 total control genes are used. (0:00:07)
computing score 'score_END'
    finished: added
    'score_END', score of gene set (adata.obs).
    50 total control genes are used. (0:00:07)
computing score 'score_MES'
    finished: added
    'score_MES', score of gene set (adata.obs).
    50 total control genes are used. (0:00:07)
computing score 'score_SMC'
    finished: added
    'score_SMC', score of gene set (adata.obs).
    150 total control genes are used. (0:00:07)
computing score 'score_IMM'
    finished: added
    'score_IMM', score of gene set (adata.obs).
    200 total control genes are used. (0:00

In [6]:
import matplotlib.pyplot as plt

sc.pl.umap(
    adata,
    color=["cell_type_major", "cell_type_score_margin", "leiden"],
    ncols=1,
    wspace=0.35,
    save="_merged_cell_type_major.pdf",
    show=False,
)
plt.show()

ct_tab = adata.obs["cell_type_major"].value_counts().rename_axis("cell_type").reset_index(name="n_cells")
ct_tab.to_csv(os.path.join(TAB_DIR, "cell_type_major_counts.csv"), index=False)
print("已写入:", os.path.join(TAB_DIR, "cell_type_major_counts.csv"))

已写入: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/tables/cell_type_major_counts.csv


In [7]:
import matplotlib.pyplot as plt

# 仅 PEC：从 raw 取全基因表达矩阵，在 PEC 内重新 HVG + 降维 + 聚类
PEC_CLUSTER_KEY = "leiden_pec_r08"  # 亚群过碎可改为 leiden_pec_r04

pec_mask = adata.obs["cell_type_major"] == "PEC"
n_pec = int(pec_mask.sum())
print("PEC 细胞数:", n_pec)
if n_pec < 50:
    raise RuntimeError("PEC 细胞过少，请检查注释或放宽 score_genes 基因集")

Xpec = adata.raw[pec_mask].X
obs_pec = adata.obs.loc[pec_mask].copy()
var_full = adata.raw.var.copy()
adata_pec = sc.AnnData(X=Xpec, obs=obs_pec, var=var_full)
adata_pec.var_names_make_unique()

sc.pp.highly_variable_genes(
    adata_pec,
    n_top_genes=2500,
    batch_key="batch" if "batch" in adata_pec.obs else None,
    flavor="seurat",
    subset=False,
)
adata_pec.raw = adata_pec.copy()
adata_pec = adata_pec[:, adata_pec.var["highly_variable"]].copy()
sc.pp.scale(adata_pec, max_value=10, zero_center=False)
sc.tl.pca(adata_pec, svd_solver="arpack")
sc.pp.neighbors(adata_pec, n_neighbors=15, n_pcs=min(40, adata_pec.obsm["X_pca"].shape[1]))
sc.tl.umap(adata_pec)
for res, key in [(0.4, "leiden_pec_r04"), (0.8, "leiden_pec_r08")]:
    sc.tl.leiden(adata_pec, resolution=res, key_added=key, flavor="igraph", n_iterations=2)

print("PEC 亚群数 (%s):" % PEC_CLUSTER_KEY, adata_pec.obs[PEC_CLUSTER_KEY].nunique())

sc.pl.umap(
    adata_pec,
    color=[PEC_CLUSTER_KEY, "batch", "Sample"],
    wspace=0.35,
    save="_pec_subclusters.pdf",
    show=False,
)
plt.show()

PEC 细胞数: 2162
extracting highly variable genes
    finished (0:00:02)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)
... be careful when using `max_value` without `zero_center`.
computing PCA
    with n_comps=50
    finished (0:00:00)
computing neighbors
    using 'X_pca' with n_pcs = 40
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:00)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:02)
running Leiden clustering
    finished: found 15 clusters and added
    'leiden_pec_r04', the cluster labels (adata.obs, categorical) (0:00:00)
running Leiden clustering
    finished: found 19 clusters and added
    'leiden_pec_r08', the cluster labels (adata.obs, 

In [8]:
import matplotlib.pyplot as plt

# PEC 亚群差异基因（相对其余 PEC）；使用 raw 全基因空间
sc.tl.rank_genes_groups(
    adata_pec,
    groupby=PEC_CLUSTER_KEY,
    method="wilcoxon",
    use_raw=True,
    key_added="rank_genes_pec",
)
sc.pl.rank_genes_groups_dotplot(
    adata_pec,
    n_genes=6,
    key="rank_genes_pec",
    save="_pec_top_markers_dotplot.pdf",
    show=False,
)
plt.show()

# 导出每个亚群 top 标志基因表（Ensembl ID）
rows = []
for g in adata_pec.obs[PEC_CLUSTER_KEY].cat.categories:
    df = sc.get.rank_genes_groups_df(adata_pec, group=g, key="rank_genes_pec")
    df = df[df["pvals_adj"] < 0.05].head(100)
    df.insert(0, "cluster", g)
    rows.append(df)
markers_pec = pd.concat(rows, axis=0)
mk_path = os.path.join(TAB_DIR, "PEC_subcluster_markers_wilcox.csv")
markers_pec.to_csv(mk_path, index=False)
print("已写入:", mk_path)

ranking genes
    finished: added to `.uns['rank_genes_pec']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:12)
    using 'X_pca' with n_pcs = 50
Storing dendrogram info using `.uns['dendrogram_leiden_pec_r08']`
已写入: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/tables/PEC_subcluster_markers_wilcox.csv


In [9]:
import matplotlib.pyplot as plt

# 谱系倾向：足细胞 vs 肾小管模块分在 PEC 亚群中的分布（来自全数据 score_genes）
for c in ["score_POD", "score_TEC"]:
    if c not in adata_pec.obs:
        print("缺少列", c, "请确认已运行注释单元")

sc.pl.umap(
    adata_pec,
    color=["score_POD", "score_TEC", PEC_CLUSTER_KEY],
    ncols=1,
    save="_pec_lineage_scores.pdf",
    show=False,
)
plt.show()

lineage_sum = (
    adata_pec.obs.groupby(PEC_CLUSTER_KEY)[["score_POD", "score_TEC"]]
    .mean()
    .rename(columns={"score_POD": "mean_POD", "score_TEC": "mean_TEC"})
)
lineage_sum["POD_minus_TEC"] = lineage_sum["mean_POD"] - lineage_sum["mean_TEC"]
lineage_sum = lineage_sum.sort_values("POD_minus_TEC", ascending=False)
print(lineage_sum.to_string())
lineage_sum.to_csv(os.path.join(TAB_DIR, "PEC_subcluster_mean_lineage_scores.csv"))

                mean_POD  mean_TEC  POD_minus_TEC
leiden_pec_r08                                   
3              -0.095092 -0.055287      -0.039805
11             -0.177664 -0.000365      -0.177299
6              -0.241269  0.017357      -0.258626
5              -0.440372 -0.110932      -0.329440
17             -0.470060 -0.103355      -0.366704
9              -0.545052 -0.140956      -0.404095
16             -0.545955 -0.115804      -0.430151
8              -0.538331 -0.095010      -0.443321
12             -0.529499 -0.057124      -0.472375
18             -0.471577  0.010630      -0.482207
13             -0.623248 -0.126971      -0.496277
10             -0.555537 -0.022983      -0.532554
0              -0.334581  0.218237      -0.552818
4              -0.514943  0.124797      -0.639740
2              -0.478409  0.287459      -0.765869
15             -0.662868  0.396460      -1.059329
7              -0.442831  0.628750      -1.071581
1              -0.591258  0.740682      -1.331941


/tmp/ipykernel_2608562/2108693669.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata_pec.obs.groupby(PEC_CLUSTER_KEY)[["score_POD", "score_TEC"]]


In [11]:
# GO 富集（Enrichr，小鼠）：各 PEC 亚群 top 上调基因 -> 基因 symbol -> GO Biological Process
import subprocess
import sys

try:
    import gseapy as gp
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gseapy", "-q"])
    import gseapy as gp

mg_go = mygene.MyGeneInfo()


def ensembl_to_symbols(ens_list, batch=400):
    out = {}
    ids = [str(x) for x in ens_list if str(x)]
    for i in range(0, len(ids), batch):
        chunk = ids[i : i + batch]
        res = mg_go.querymany(
            chunk, scopes="ensembl.gene", fields="symbol", species="mouse", verbose=False
        )
        for r in res:
            q = r.get("query")
            sym = r.get("symbol")
            if q and sym:
                out[q] = sym
    return out


GO_LIB = "GO_Biological_Process_2023"
pec_clusters = list(adata_pec.obs[PEC_CLUSTER_KEY].astype(str).unique())
go_outdir = os.path.join(TAB_DIR, "PEC_GO_enrichr")
os.makedirs(go_outdir, exist_ok=True)

for g in pec_clusters:
    df = sc.get.rank_genes_groups_df(adata_pec, group=g, key="rank_genes_pec")
    df = df[(df["pvals_adj"] < 0.05) & (df["logfoldchanges"] > 0.25)].head(200)
    ens = df["names"].astype(str).tolist()
    sym_map = ensembl_to_symbols(ens)
    genes = sorted({sym_map[e] for e in ens if e in sym_map})
    if len(genes) < 10:
        print("簇 %s 可用 symbol 基因过少 (%d)，跳过 GO" % (g, len(genes)))
        continue
    try:
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=[GO_LIB],
            organism="mouse",
            outdir=os.path.join(go_outdir, f"cluster_{g}"),
            no_plot=True,
            verbose=False,
        )
        res_df = enr.results
        res_path = os.path.join(go_outdir, f"GO_BP_cluster_{g}.csv")
        res_df.to_csv(res_path, index=False)
        print("簇", g, "GO 结果:", res_path, "命中条数", len(res_df))
    except Exception as ex:
        print("簇", g, "GO 失败:", ex)

print("GO 输出目录:", go_outdir)

簇 0 GO 结果: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/tables/PEC_GO_enrichr/GO_BP_cluster_0.csv 命中条数 1347
簇 1 GO 结果: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/tables/PEC_GO_enrichr/GO_BP_cluster_1.csv 命中条数 1334
簇 2 GO 结果: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/tables/PEC_GO_enrichr/GO_BP_cluster_2.csv 命中条数 1503
簇 3 GO 结果: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/tables/PEC_GO_enrichr/GO_BP_cluster_3.csv 命中条数 1596
簇 4 GO 结果: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/tables/PEC_GO_enrichr/GO_BP_cluster_4.csv 命中条数 1385
簇 5 GO 结果: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/tables/PEC_GO_enrichr/GO_BP_cluster_5.csv 命中条数 1791
簇 6 GO 结果: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/tables/PEC_GO_enrichr/GO_BP_cluster_6.csv 命中条数 1480
簇 7 GO 结果: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/tables/PEC_GO_enrichr/GO_BP_cluster_7.csv 命中条数 1172
簇 8 GO 结果: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run

In [12]:
out_h5ad = os.path.join(OUT_DIR, "GSE146912_merged_analyzed.h5ad")
adata.write_h5ad(out_h5ad, compression="gzip")
print("已保存（含 cell_type_major 等注释）:", out_h5ad)

out_pec = os.path.join(OUT_DIR, "GSE146912_PEC_reclustered.h5ad")
adata_pec.write_h5ad(out_pec, compression="gzip")
print("已保存 PEC 子集重聚类:", out_pec)

print(f"剩余空间: {shutil.disk_usage(OUT_DIR).free / (1024**3):.1f} GiB")

已保存（含 cell_type_major 等注释）: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/GSE146912_merged_analyzed.h5ad
已保存 PEC 子集重聚类: /mnt/newdisk/GDN1/scRNA_Project/results/GSE146912_run/GSE146912_PEC_reclustered.h5ad
剩余空间: 698.6 GiB
